In [2]:


import sys
sys.path.insert(0, "/home/anubhav/development/vla_from_scratch")  # your project root

In [3]:
# Cell — imports for modular profiling
import torch
from torch.utils.flop_counter import FlopCounterMode
import time

from encoders.vision.resnet       import VisionEncoderResnet18
from encoders.vision.efficientnet import VisionEncoderEfficientNet
from encoders.text.distilbert     import TextEncoderDistilbert
from encoders.text.smollm         import TextEncoderSmolLM
from encoders.state.mlp           import StateEncoderMLP
from utils.tokenizer              import tokenize_instruction

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
B           = 64
OBS_HORIZON = 2
STATE_DIM   = 39
D_MODEL     = 256

image     = torch.randn(B, OBS_HORIZON * 3, 224, 224).to(DEVICE)
state     = torch.randn(B, STATE_DIM).to(DEVICE)
ids, mask = tokenize_instruction("reach the target")
text_ids  = ids.repeat(B, 1).to(DEVICE)
text_mask = mask.repeat(B, 1).to(DEVICE)

print("Setup done")

/home/anubhav/anaconda3/envs/easyvla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup done


In [4]:

# Cell — timing utility
def benchmark(fn, device, n=20):
    if device == "cuda":
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        for _ in range(3): fn()   # warmup
        torch.cuda.synchronize()
        start.record()
        for _ in range(n): fn()
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / n
    else:
        for _ in range(3): fn()
        t0 = time.perf_counter()
        for _ in range(n): fn()
        return (time.perf_counter() - t0) / n * 1000

def gpu_mem(fn, device):
    if device != "cuda":
        return 0.0
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        fn()
    return torch.cuda.max_memory_allocated() / 1024**3

def flops(fn):
    with FlopCounterMode(display=False) as fc:
        fn()
    return fc.get_total_flops() / 1e9

print("Utilities ready")

Utilities ready


In [5]:
# Cell — vision encoder comparison
print(f"── Vision Encoders (batch={B}, obs_horizon={OBS_HORIZON}) ──")
print(f"{'Encoder':<18} {'Time (ms)':>10} {'Memory (GB)':>12} {'GFLOPs':>10} {'Params (M)':>12} {'Trainable (M)':>14}")
print("-" * 80)

for name, cls in [
    ("resnet18",     VisionEncoderResnet18),
    ("efficientnet", VisionEncoderEfficientNet),
]:
    enc = cls(d_model=D_MODEL, obs_horizon=OBS_HORIZON).to(DEVICE)
    enc.eval()

    fn = lambda: enc(image)

    with torch.no_grad():
        ms  = benchmark(fn, DEVICE)
        mem = gpu_mem(fn, DEVICE)
        gf  = flops(fn)

    total     = sum(p.numel() for p in enc.parameters()) / 1e6
    trainable = sum(p.numel() for p in enc.parameters() if p.requires_grad) / 1e6

    print(f"  {name:<16} {ms:>10.1f} {mem:>12.3f} {gf:>10.3f} {total:>12.2f} {trainable:>14.2f}")
    del enc
    torch.cuda.empty_cache()

── Vision Encoders (batch=64, obs_horizon=2) ──
Encoder             Time (ms)  Memory (GB)     GFLOPs   Params (M)  Trainable (M)
--------------------------------------------------------------------------------
  resnet18              101.7        0.505    248.064        11.32          10.63
  efficientnet           86.0        0.719     52.663         4.34           0.74


In [6]:
# Cell — text encoder comparison
print(f"── Text Encoders (batch={B}) ──")
print(f"{'Encoder':<18} {'Time (ms)':>10} {'Memory (GB)':>12} {'GFLOPs':>10} {'Params (M)':>12} {'Trainable (M)':>14}")
print("-" * 80)

for name, cls in [
    ("distilbert", TextEncoderDistilbert),
    ("smollm",     TextEncoderSmolLM),
]:
    enc = cls(d_model=D_MODEL).to(DEVICE)
    enc.eval()

    fn = lambda: enc(text_ids, text_mask)

    with torch.no_grad():
        ms  = benchmark(fn, DEVICE)
        mem = gpu_mem(fn, DEVICE)
        gf  = flops(fn)

    total     = sum(p.numel() for p in enc.parameters()) / 1e6
    trainable = sum(p.numel() for p in enc.parameters() if p.requires_grad) / 1e6

    print(f"  {name:<16} {ms:>10.1f} {mem:>12.3f} {gf:>10.3f} {total:>12.2f} {trainable:>14.2f}")
    del enc
    torch.cuda.empty_cache()

── Text Encoders (batch=64) ──
Encoder             Time (ms)  Memory (GB)     GFLOPs   Params (M)  Trainable (M)
--------------------------------------------------------------------------------


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12839.97it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  distilbert             52.0        0.391    164.891        66.56           0.20


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4318.66it/s]


  smollm                161.5        0.728    412.234       134.66           0.15


In [7]:
# Cell — full model comparison (all combos)
from vla_diffusion import VlaDiffusion

ACTION_HORIZON = 8
action = torch.randn(B, ACTION_HORIZON, 4).to(DEVICE)

print(f"── Full Model Combos (batch={B}) ──")
print(f"{'Vision':<15} {'Text':<12} {'Train ms':>10} {'Infer ms':>10} {'Mem (GB)':>10} {'Params (M)':>12}")
print("-" * 75)

for vis, txt in [
    ("resnet18",     "distilbert"),
    ("resnet18",     "smollm"),
    ("efficientnet", "distilbert"),
    ("efficientnet", "smollm"),
]:
    model = VlaDiffusion(
        state_dim      = STATE_DIM,
        action_dim     = 4,
        d_model        = D_MODEL,
        action_horizon = ACTION_HORIZON,
        obs_horizon    = OBS_HORIZON,
        vision_encoder = vis,
        text_encoder   = txt,
    ).to(DEVICE)

    # training step time (with grad)
    def train_fn():
        loss = model.loss(image, text_ids, text_mask, state, action)
        loss.backward()
        model.zero_grad()

    # inference time (no grad)
    def infer_fn():
        with torch.no_grad():
            model.act(image, text_ids, text_mask, state)

    train_ms = benchmark(train_fn, DEVICE, n=10)  # fewer runs for train
    infer_ms = benchmark(infer_fn, DEVICE, n=20)
    mem      = gpu_mem(lambda: model.loss(image, text_ids, text_mask, state, action), DEVICE)
    params   = sum(p.numel() for p in model.parameters()) / 1e6

    print(f"  {vis:<13} {txt:<12} {train_ms:>10.1f} {infer_ms:>10.1f} {mem:>10.3f} {params:>12.2f}")
    del model
    torch.cuda.empty_cache()

── Full Model Combos (batch=64) ──
Vision          Text           Train ms   Infer ms   Mem (GB)   Params (M)
---------------------------------------------------------------------------


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7492.24it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/anubhav/development/vla_from_scratch/fusion.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.context_encoder = nn.TransformerEncoder(self_layer, num_layers=n_layers)


  resnet18      distilbert        223.7      135.9      0.793        85.62


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4417.13it/s]


  resnet18      smollm            225.7      136.8      1.055       153.72


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12921.06it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  efficientnet  distilbert        176.5      124.9      1.007        78.63


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4753.84it/s]


  efficientnet  smollm            177.4      125.2      1.267       146.74


In [8]:
# Cell — text cache impact (shows how much the cache saves)
print("── Text Cache Impact ──")

model = VlaDiffusion(
    state_dim=STATE_DIM, action_dim=4,
    d_model=D_MODEL, action_horizon=ACTION_HORIZON,
    obs_horizon=OBS_HORIZON,
).to(DEVICE)
model.eval()

# without cache — clear it each time
def no_cache():
    model._text_cache = {}
    with torch.no_grad():
        model.act(image, text_ids, text_mask, state)

# with cache — populated after first call
model.act(image, text_ids, text_mask, state)  # populate cache
def with_cache():
    with torch.no_grad():
        model.act(image, text_ids, text_mask, state)

ms_no_cache   = benchmark(no_cache,   DEVICE)
ms_with_cache = benchmark(with_cache, DEVICE)
saving        = ms_no_cache - ms_with_cache

print(f"  Without cache:  {ms_no_cache:.1f} ms")
print(f"  With cache:     {ms_with_cache:.1f} ms")
print(f"  Saving:         {saving:.1f} ms  ({100*saving/ms_no_cache:.0f}% faster)")
del model

── Text Cache Impact ──


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10770.64it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Without cache:  182.9 ms
  With cache:     130.7 ms
  Saving:         52.2 ms  (29% faster)
